# BUG: `DatetimeIndex` slicing: `df.loc["2020-01-01": "2020-01-01"]` gives different result than `df.loc[Timestamp("2020-01-01"): Timestamp("2020-01-01")]` 

One would expect that both times the same data is selected. It is debatable which one of the two results is the "correct" one; essentially this is the question of whether one wants an inclusive range or not by default.

In any case I would definitely call it a bug though that both calls return different slices.  It is just way too easy to trip over this behavior and accidentally get a wrong slice!

The reason for the different slices seems to be that using the string or `Timestamp` return different things:

- `df.index.get_loc(a) →slice(0, 96, None) `
- `df.index.get_loc(ta) →0 `
- `df.index.get_loc(b) →slice(192, 288, None) `
- `df.index.get_loc(tb) →192 `

In particular, `df.loc["2020-01-01"]` returns all data collected on January first, whereas `df.loc[Timestamp("2020-01-01")]` returns only the single datapoint collected at `2020-01-01 00:00:00`.

The [documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#indexing) says that

> This type of slicing will work on a DataFrame with a DatetimeIndex as well. Since the partial string selection is a form of label slicing, the endpoints will be included. This would include matching times on an included date

In [16]:
from pandas import date_range, Timestamp, DataFrame, Period

ts = date_range(start="2020-01-01", end="2020-02-01", freq="15min")
df = DataFrame(range(len(ts)), index=ts)

a = "2020-01-01"
b = "2020-01-03"
ta = Timestamp(a)
tb = Timestamp(b)

print(df.index.get_loc(a), df.index.get_loc(ta))
print(df.index.get_loc(b), df.index.get_loc(tb))
print(df.loc[a:b])
print(df.loc[ta: tb])

slice(0, 96, None) 0
slice(192, 288, None) 192
                       0
2020-01-01 00:00:00    0
2020-01-01 00:15:00    1
2020-01-01 00:30:00    2
2020-01-01 00:45:00    3
2020-01-01 01:00:00    4
...                  ...
2020-01-03 22:45:00  283
2020-01-03 23:00:00  284
2020-01-03 23:15:00  285
2020-01-03 23:30:00  286
2020-01-03 23:45:00  287

[288 rows x 1 columns]
                       0
2020-01-01 00:00:00    0
2020-01-01 00:15:00    1
2020-01-01 00:30:00    2
2020-01-01 00:45:00    3
2020-01-01 01:00:00    4
...                  ...
2020-01-02 23:00:00  188
2020-01-02 23:15:00  189
2020-01-02 23:30:00  190
2020-01-02 23:45:00  191
2020-01-03 00:00:00  192

[193 rows x 1 columns]


In [25]:
str(Timestamp(a))

'2020-01-01 00:00:00'

In [23]:
print(df[a:b])
print(df[ta: tb])

                       0
2020-01-01 00:00:00    0
2020-01-01 00:15:00    1
2020-01-01 00:30:00    2
2020-01-01 00:45:00    3
2020-01-01 01:00:00    4
...                  ...
2020-01-03 22:45:00  283
2020-01-03 23:00:00  284
2020-01-03 23:15:00  285
2020-01-03 23:30:00  286
2020-01-03 23:45:00  287

[288 rows x 1 columns]
                       0
2020-01-01 00:00:00    0
2020-01-01 00:15:00    1
2020-01-01 00:30:00    2
2020-01-01 00:45:00    3
2020-01-01 01:00:00    4
...                  ...
2020-01-02 23:00:00  188
2020-01-02 23:15:00  189
2020-01-02 23:30:00  190
2020-01-02 23:45:00  191
2020-01-03 00:00:00  192

[193 rows x 1 columns]


In [21]:
Period("2020-01-01 to 2020-01-03")

DateParseError: Unknown datetime string format, unable to parse: 2020-01-01 TO 2020-01-03

In [15]:
df.loc[Timestamp("2020-01-01")]

0    0
Name: 2020-01-01 00:00:00, dtype: int64

In [11]:
df.index

DatetimeIndex(['2020-01-01 00:00:00', '2020-01-01 00:15:00',
               '2020-01-01 00:30:00', '2020-01-01 00:45:00',
               '2020-01-01 01:00:00', '2020-01-01 01:15:00',
               '2020-01-01 01:30:00', '2020-01-01 01:45:00',
               '2020-01-01 02:00:00', '2020-01-01 02:15:00',
               ...
               '2020-01-31 21:45:00', '2020-01-31 22:00:00',
               '2020-01-31 22:15:00', '2020-01-31 22:30:00',
               '2020-01-31 22:45:00', '2020-01-31 23:00:00',
               '2020-01-31 23:15:00', '2020-01-31 23:30:00',
               '2020-01-31 23:45:00', '2020-02-01 00:00:00'],
              dtype='datetime64[ns]', length=2977, freq='15T')

In [9]:
import pandas
pandas.show_versions()


INSTALLED VERSIONS
------------------
commit           : 73c68257545b5f8530b7044f56647bd2db92e2ba
python           : 3.9.7.final.0
python-bits      : 64
OS               : Linux
OS-release       : 5.13.0-20-generic
Version          : #20-Ubuntu SMP Fri Oct 15 14:21:35 UTC 2021
machine          : x86_64
processor        : x86_64
byteorder        : little
LC_ALL           : None
LANG             : en_US.UTF-8
LOCALE           : en_US.UTF-8

pandas           : 1.3.3
numpy            : 1.21.3
pytz             : 2021.3
dateutil         : 2.8.2
pip              : 21.3.1
setuptools       : 58.2.0
Cython           : None
pytest           : 6.2.5
hypothesis       : None
sphinx           : 4.2.0
blosc            : None
feather          : None
xlsxwriter       : None
lxml.etree       : None
html5lib         : 1.1
pymysql          : None
psycopg2         : None
jinja2           : 3.0.2
IPython          : 7.28.0
pandas_datareader: None
bs4              : 4.10.0
bottleneck       : None
fsspec      